# pm_helper: PowerModels.jl power flow solver
* conda: h-py312-basic
* notes and design details: [`work-notes/pm_helper.md`](../work-notes/pm_helper.md) (TODO)

Simple sanity-check notebook for the PowerModels.jl driver (`pm_solve.jl`), mirroring
the style of [`gridkit_helper.ipynb`](gridkit_helper.ipynb): one base case, one solve,
inspect the result. `pm_solve.jl` is a thin Julia wrapper around PowerModels.jl —
called via `subprocess`, same pattern as `solve_pf` in [`pf_helper.ipynb`](pf_helper.ipynb).

## workflow
1. **Section 1**: imports and path setup
2. **Section 2**: parse the base case and inspect the network dict
3. **Section 3**: solve AC power flow, display results
4. **Section 4**: solve DC power flow, compare vs AC
5. **Section 5**: base case, three-way comparison (PM.jl AC, GridKit `solve_pf`, TAMU/PowerWorld reference)
6. **Section 6**: perturbed-case sweep, PM.jl vs GridKit `solve_pf` on all 14 existing cases
7. **Section 7**: data-driven verdict - is GridKit's `solve_pf` reasonable for the Aleatoric UQ track?


In [8]:
import os
import sys
import json
import importlib
from pathlib import Path

import pandas as pd
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.6f}".format)

# === GridKit / pm-solver paths ===
GRIDKIT_REPO = Path.home() / "gridkit"
UQ_DIR = GRIDKIT_REPO / "uq-usecase"
PM_PROJECT_DIR = UQ_DIR / "pm-solver"
PM_SOLVE_JL = PM_PROJECT_DIR / "pm_solve.jl"
JULIA_BIN = Path.home() / "bin/julia112"

# === case data: reuse the .m files already generated for pf_helper.ipynb ===
PF_M_CASES = UQ_DIR / "pf-solver/m-cases"
BASECASE_M = PF_M_CASES / "basecase/case_ACTIVSg200.m"

# === py-utils on path ===
PY_UTILS_DIR = str(UQ_DIR / "py-utils")
if PY_UTILS_DIR not in sys.path:
    sys.path.insert(0, PY_UTILS_DIR)

import pf_utils, pm_utils

importlib.reload(pf_utils)
importlib.reload(pm_utils)
from pm_utils import run_pm_solve, run_pm_solve_out, pm_summary

for label, p in [
    ("julia112 binary", JULIA_BIN),
    ("pm-solver Project.toml", PM_PROJECT_DIR / "Project.toml"),
    ("pm_solve.jl", PM_SOLVE_JL),
    ("base case .m", BASECASE_M),
    ("pm_utils.py", UQ_DIR / "py-utils/pm_utils.py"),
]:
    status = "OK " if p.exists() else "MISSING"
    print(f"  [{status}]  {label}: {p}")

<module 'pf_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/pf_utils.py'>

<module 'pm_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/pm_utils.py'>

  [OK ]  julia112 binary: /home/isatkaus/bin/julia112
  [OK ]  pm-solver Project.toml: /home/isatkaus/gridkit/uq-usecase/pm-solver/Project.toml
  [OK ]  pm_solve.jl: /home/isatkaus/gridkit/uq-usecase/pm-solver/pm_solve.jl
  [OK ]  base case .m: /home/isatkaus/gridkit/uq-usecase/pf-solver/m-cases/basecase/case_ACTIVSg200.m
  [OK ]  pm_utils.py: /home/isatkaus/gridkit/uq-usecase/py-utils/pm_utils.py


# Section 2: parse the base case and inspect the network dict

`pm_solve.jl` parses the `.m` file internally via `PowerModels.parse_file()` and prints
a one-line summary to stderr (bus/gen counts, offline gens, baseMVA). Run it once here
just to confirm the parse succeeds and see that summary, before solving anything.


In [9]:
import subprocess

# Small inline Julia snippet: parse the case and print bus/gen/branch/load counts
# as JSON, so we can inspect the network dict without a full solve.
_inspect_jl = f"""
using PowerModels, JSON
PowerModels.silence()
data = PowerModels.parse_file("{BASECASE_M}")
summary = Dict(
    "n_bus" => length(data["bus"]),
    "n_gen" => length(data["gen"]),
    "n_branch" => length(data["branch"]),
    "n_load" => length(data["load"]),
    "n_gen_offline" => count(g -> g["gen_status"] == 0, values(data["gen"])),
    "baseMVA" => data["baseMVA"],
)
println(JSON.json(summary))
"""

r = subprocess.run(
    [str(JULIA_BIN), f"--project={PM_PROJECT_DIR}", "-e", _inspect_jl],
    capture_output=True,
    text=True,
)
print(r.stderr[-500:] if r.returncode != 0 else "")
network_summary = json.loads(r.stdout.strip().splitlines()[-1])
network_summary

{'baseMVA': 100,
 'n_branch': 245,
 'n_bus': 200,
 'n_gen': 49,
 'n_gen_offline': 11,
 'n_load': 108}

# Section 3: solve AC power flow

## How the Python notebook calls Julia

Each call to `run_pm_solve` / `run_pm_solve_out` in `pm_utils.py` launches a fresh
`subprocess`:

```
julia112 --project=<pm-solver/> pm_solve.jl <input.m> [--output-m ...] [--tol ...] [--max-iter ...] [--flat-start]
```

- **`--project=pm-solver/`**: tells Julia to load exactly the packages pinned in
  `pm-solver/Project.toml` (PowerModels v0.21.6, Ipopt v1.15.0, JuMP v1.31.1).
  This is the Julia equivalent of a conda environment.
- **Precompilation**: Julia compiles each package to native code on first load and
  stores the result in `~/.julia/compiled/`. Subsequent calls reuse this cache, but
  the cache is reloaded from disk on every subprocess invocation.
- **Per-call overhead (measured on this node)**: ~8.5 s total wall time, of which
  ~5.5 s is Julia startup + loading the precompiled packages, and ~2.9 s is Ipopt
  solving the 200-bus AC PF. For the 14-case sweep this is ~2 min total. For 8760
  scenarios this cost would be prohibitive; the plan is a batched manifest mode
  where one Julia process loads the packages once and loops over all cases (see
  `plan.md` Phase D).
- **stdout / stderr split**: `pm_solve.jl` prints one `bus <i> V=... theta_deg=...
  type=...` line per bus to stdout; all diagnostics (solver params, Ipopt summary,
  termination status) go to stderr. `subprocess.run(capture_output=True)` keeps them
  separate.

## Warm start vs. cold start (`--flat-start`)

By default this is a **warm start**: `PowerModels.parse_file` reads the Vm/Va values
already embedded in the `.m` file into the data dict, and Ipopt uses those as its
initial point. For the raw base case (`case_ACTIVSg200.m`) those are the
TAMU/PowerWorld solved values (Vm≈1.01–1.05 pu, Va≈-3 to -11°), so Ipopt
converges in only 4 iterations.

For the perturbed cases the same TAMU Vm/Va are the initial point (the perturbation
scripts only modify PD/QD/PG, not bus Vm/Va), so it's a warm start from the
unperturbed solution — still close for mild perturbations, potentially farther for
stress cases like `load80pct`.

**`--flat-start`** (available via `flat_start=True` in `run_pm_solve`) resets all
buses to Vm=1.0, Va=0.0 before solving. This is useful to:
1. Check robustness: can PM.jl find the solution without a near-initial-point?
2. Match GridKit's `--flat-start` behavior for a fair head-to-head comparison.
3. Simulate the 8760 production scenario where PCM hourly `.m` files may not have
   reliable embedded Vm/Va values.


In [7]:
# Solver parameters: adjust tol/max_iter here to experiment before running the sweep.
IPOPT_TOL = 1e-8  # Ipopt default; tighten to 1e-10 if solutions look borderline
IPOPT_MAX_ITER = 300

# Persist the solved .m alongside the base case, mirroring pf-solver/m-cases/'s
# <name>_solved.m convention (PM.jl's own m-cases tree).
PM_M_CASES = PM_PROJECT_DIR / "m-cases"
PM_BASECASE_SOLVED_M = PM_M_CASES / "basecase/case_ACTIVSg200_solved.m"
PM_BASECASE_SOLVED_M.parent.mkdir(parents=True, exist_ok=True)

ac_df, ac_stderr, ac_rc = run_pm_solve_out(
    JULIA_BIN,
    PM_PROJECT_DIR,
    PM_SOLVE_JL,
    BASECASE_M,
    PM_BASECASE_SOLVED_M,
    pf_type="ac",
    tol=IPOPT_TOL,
    max_iter=IPOPT_MAX_ITER,
)
pm_summary("ACTIVSg200 base case (PM.jl AC PF)", ac_df, ac_rc, ac_stderr)

# --- sanity checks vs TAMU/PowerWorld reference embedded in the raw .m ---
# The raw case file ships with Vm/Va already solved by the data provider (TAMU/PowerWorld);
# these are the closest thing to a "ground truth" for this synthetic network.
if ac_rc == 0:
    from pf_utils import parse_raw_m_bus, diff_vs_base

    tamu_df = parse_raw_m_bus(BASECASE_M)
    print("Sanity check: PM.jl vs TAMU/PowerWorld reference (embedded in raw .m):")
    _ = diff_vs_base("PM.jl vs TAMU", ac_df, tamu_df)

ac_df.head()

ACTIVSg200 base case (PM.jl AC PF)
  tol=1.0e-8  max_iter=300
  termination_status=LOCALLY_SOLVED
  solve_time=2.936875820159912s
  converged=true
  ipopt_iters=4  final_nlp_error=n/a
  CONVERGED   rc=0   buses=200
  V range: [1.0102, 1.0556] pu   violations (V<0.95 or V>1.05): 1

Sanity check: PM.jl vs TAMU/PowerWorld reference (embedded in raw .m):
  vs base-case solution (PM.jl vs TAMU):
    max |dV|      = 0.000011 pu
    mean |dV|     = 0.000002 pu
    max |dTheta|  = 0.000926 deg
    mean |dTheta| = 0.000460 deg



,bus_i,V_pu,theta_deg,type
0,1,1.019154,-7.085952,1
1,2,1.019036,-7.098785,1
2,3,1.030059,-10.029511,1
3,4,1.030034,-10.032395,1
4,5,1.037275,-3.546718,1


# Section 4: solve DC power flow, compare vs AC

DC power flow is a linearized, real-power-only approximation: voltage magnitude is
fixed at 1.0 p.u. everywhere and only angle is solved. It's much faster (no reactive
power, no Newton iterations on a nonlinear model) and is the same approximation
discussed in [`pf_helper.md` Section 13](../work-notes/pf_helper.md) as being
consistent with GridKit's angle-dominated PF response on this network.


In [4]:
dc_df, dc_stderr, dc_rc = run_pm_solve(
    JULIA_BIN, PM_PROJECT_DIR, PM_SOLVE_JL, BASECASE_M, pf_type="dc"
)
pm_summary("ACTIVSg200 base case (DC PF)", dc_df, dc_rc, dc_stderr)
dc_df.head()

ACTIVSg200 base case (DC PF)
  [pm_solve] parsed /home/isatkaus/gridkit/uq-usecase/pf-solver/m-cases/basecase/case_ACTIVSg200.m: 200 buses, 49 gens (11 offline), baseMVA=100
  [pm_solve] solver=ipopt  pf_type=dc
  [pm_solve] termination_status=LOCALLY_SOLVED
  [pm_solve] solve_time=0.011840105056762695s
  [pm_solve] converged=true
  CONVERGED   rc=0   buses=200
  V range: [1.0000, 1.0000] pu   violations (V<0.95 or V>1.05): 0



,bus_i,V_pu,theta_deg,type
0,1,1.000000,-7.514640,1
1,2,1.000000,-7.529352,1
2,3,1.000000,-10.733105,1
3,4,1.000000,-10.736419,1
4,5,1.000000,-3.654975,1


In [5]:
cmp = ac_df.merge(
    dc_df[["bus_i", "theta_deg"]].rename(columns={"theta_deg": "theta_deg_dc"}),
    on="bus_i",
)
cmp["dTheta_ac_vs_dc"] = (cmp.theta_deg - cmp.theta_deg_dc).abs()

print("AC vs DC angle comparison (base case):")
print(f"  max |theta_AC - theta_DC| = {cmp.dTheta_ac_vs_dc.max():.4f} deg")
print(f"  mean |theta_AC - theta_DC| = {cmp.dTheta_ac_vs_dc.mean():.4f} deg")
print(f"  AC voltage range: [{ac_df.V_pu.min():.4f}, {ac_df.V_pu.max():.4f}] pu")
print("  DC voltage: fixed at 1.0 pu everywhere (not modeled)")
cmp.sort_values("dTheta_ac_vs_dc", ascending=False).head(10)

AC vs DC angle comparison (base case):
  max |theta_AC - theta_DC| = 0.7828 deg
  mean |theta_AC - theta_DC| = 0.3978 deg
  AC voltage range: [1.0102, 1.0556] pu
  DC voltage: fixed at 1.0 pu everywhere (not modeled)


,bus_i,V_pu,theta_deg,type,theta_deg_dc,dTheta_ac_vs_dc
61,62,1.026724,-11.318444,1,-12.101243,0.782799
144,145,1.032020,-11.164116,1,-11.919794,0.755678
159,160,1.031150,-11.239883,1,-11.989418,0.749535
160,161,1.031150,-11.239883,1,-11.989418,0.749535
158,159,1.026595,-10.742864,1,-11.486446,0.743581
180,181,1.033392,-11.195465,1,-11.927620,0.732155
41,42,1.030148,-11.070407,1,-11.801030,0.730623
56,57,1.028140,-10.449583,1,-11.177924,0.728341
39,40,1.024565,-10.496824,1,-11.223160,0.726336
38,39,1.024593,-10.493529,1,-11.219437,0.725908


# Section 5: base case, three-way comparison (PM.jl, GridKit, TAMU/PowerWorld)

Cross-validates PM.jl's AC PF solution against two independent references:
- **GridKit `solve_pf`**: solved in `pf_helper.ipynb` Section 4, persisted to
  `pf-solver/m-cases/basecase/case_ACTIVSg200_solved.m`
  (V range [1.0091, 1.0432] pu, 0 violations, nni=4, ||f||=4.42e-6).
- **TAMU/PowerWorld reference**: Vm/Va embedded directly in the raw
  `case_ACTIVSg200.m` from the data provider (see [`pf_helper.md`](../work-notes/pf_helper.md)
  Section 8). Read with no solve via `parse_raw_m_bus`.


In [ ]:
import sys

if str(UQ_DIR / "py-utils") not in sys.path:
    sys.path.insert(0, str(UQ_DIR / "py-utils"))
from pf_utils import pf_summary, diff_vs_base, parse_raw_m_bus

# GridKit's solved base case was produced in pf_helper.ipynb (see Section 4 there)
# and persisted to pf-solver/m-cases/basecase/case_ACTIVSg200_solved.m.
# Read it directly via parse_raw_m_bus - no need to re-run solve_pf here.
GRIDKIT_SOLVED_M = PF_M_CASES / "basecase/case_ACTIVSg200_solved.m"
print(
    f"  [{'OK ' if GRIDKIT_SOLVED_M.exists() else 'MISSING'}]  GridKit solved base case: {GRIDKIT_SOLVED_M}"
)

gk_df = parse_raw_m_bus(GRIDKIT_SOLVED_M)
print(
    f"  GridKit solve_pf base case: V range [{gk_df.V_pu.min():.4f}, {gk_df.V_pu.max():.4f}] pu"
)

# TAMU/PowerWorld reference: embedded in the raw unsolved case file
tamu_df = parse_raw_m_bus(BASECASE_M)
print(
    f"  TAMU/PowerWorld reference:  V range [{tamu_df.V_pu.min():.4f}, {tamu_df.V_pu.max():.4f}] pu"
)

  [OK ]  solve_pf binary: /home/isatkaus/gridkit/uq-usecase/pf-solver/build/solve_pf
ACTIVSg200 base case (GridKit solve_pf)
  CONVERGED   rc=0   ||f||=4.42166e-06   buses=200   nni=4
  V range: [1.0091, 1.0432] pu   violations (V<0.95 or V>1.05): 0



In [ ]:
print(
    "=== Base case: three-way comparison (PM.jl AC, GridKit solve_pf, TAMU/PowerWorld) ===\n"
)
pm_vs_gk = diff_vs_base("PM.jl (AC) vs GridKit solve_pf", ac_df, gk_df)
pm_vs_tamu = diff_vs_base("PM.jl (AC) vs TAMU/PowerWorld reference", ac_df, tamu_df)
gk_vs_tamu = diff_vs_base(
    "GridKit solve_pf vs TAMU/PowerWorld reference", gk_df, tamu_df
)

pm_vs_gk.sort_values("dV", ascending=False).head(10)

  vs base-case solution (PM.jl (AC) vs GridKit solve_pf):
    max |dV|      = 0.029728 pu
    mean |dV|     = 0.005057 pu
    max |dTheta|  = 0.190703 deg
    mean |dTheta| = 0.073286 deg



,bus_i,V_pu,theta_deg,type,V_base,theta_base,dV,dTheta
99,100,1.055588,-7.845709,1,1.025860,-7.802112,0.029728,0.043597
94,95,1.045270,-6.144382,1,1.019603,-6.053667,0.025666,0.090715
79,80,1.047561,-7.259080,1,1.024923,-7.241397,0.022638,0.017683
183,184,1.049219,-8.289697,1,1.027499,-8.325714,0.021721,0.036017
173,174,1.048555,-6.976316,1,1.027241,-6.965693,0.021314,0.010622
40,41,1.046279,-7.960739,1,1.026631,-8.005822,0.019647,0.045083
15,16,1.048479,-7.176377,1,1.029456,-7.185228,0.019022,0.008851
14,15,1.049105,-7.105445,1,1.030094,-7.111651,0.019010,0.006207
172,173,1.043871,-7.825538,1,1.025472,-7.873502,0.018399,0.047964
193,194,1.040085,-10.703598,1,1.022684,-10.765368,0.017400,0.061769


# Section 6: perturbed-case sweep (PM.jl vs GridKit `solve_pf`)

Repeat the comparison across the 14 already-generated perturbed cases in
[`pf-solver/m-cases`](../pf-solver/m-cases) (5 load levels, 4 wind-curtailment
levels, 5 generator-outage scenarios; documented in
[`pf_helper.md`](../work-notes/pf_helper.md) Sections 10-12). Per the plan
(`/memories/session/plan.md` Phase C), GridKit's already-solved `*_solved.m` files
are reused directly (read via `parse_raw_m_bus`, no re-solve) rather than
re-running `solve_pf`; only PM.jl needs to solve each raw case fresh.


In [ ]:
PERTURBED_CASES = [
    "gen2rand_off",
    "gen3rand_off",
    "gen5rand_off",
    "gen10rand_off",
    "gen147off",
    "load5pct",
    "load10pct",
    "load20pct",
    "load40pct",
    "load80pct",
    "wind10pct",
    "wind20pct",
    "wind40pct",
    "wind80pct",
]

sweep_rows = []
for case in PERTURBED_CASES:
    raw_m = PF_M_CASES / f"case_ACTIVSg200_{case}.m"
    gk_solved_m = PF_M_CASES / f"case_ACTIVSg200_{case}_solved.m"
    if not raw_m.exists() or not gk_solved_m.exists():
        print(f"  [SKIP] {case}: missing raw or GridKit-solved .m")
        continue

    pm_df, pm_stderr, pm_rc = run_pm_solve(
        JULIA_BIN,
        PM_PROJECT_DIR,
        PM_SOLVE_JL,
        raw_m,
        pf_type="ac",
        tol=IPOPT_TOL,
        max_iter=IPOPT_MAX_ITER,
    )
    gk_case_df = parse_raw_m_bus(gk_solved_m)

    row = {"case": case, "pm_converged": pm_rc == 0}
    if pm_rc == 0:
        import re as _re

        iters_m = _re.search(r"Number of Iterations\.+:\s*(\d+)", pm_stderr)
        row["ipopt_iters"] = int(iters_m.group(1)) if iters_m else None
        cmp = pm_df.merge(
            gk_case_df[["bus_i", "V_pu", "theta_deg"]].rename(
                columns={"V_pu": "V_gk", "theta_deg": "theta_gk"}
            ),
            on="bus_i",
        )
        cmp["dV"] = (cmp.V_pu - cmp.V_gk).abs()
        cmp["dTheta"] = (cmp.theta_deg - cmp.theta_gk).abs()
        row["max_dV_pu"] = cmp.dV.max()
        row["mean_dV_pu"] = cmp.dV.mean()
        row["max_dTheta_deg"] = cmp.dTheta.max()
        row["mean_dTheta_deg"] = cmp.dTheta.mean()
    sweep_rows.append(row)

sweep_df = pd.DataFrame(sweep_rows)
sweep_df

# Section 7: is GridKit's `solve_pf` reasonable for the Aleatoric UQ track?

Decision criteria (thresholds chosen to reflect "practically indistinguishable
for planning-level UQ", not exact numerical equality):
- **Convergence**: PM.jl should converge on all (or nearly all) of the 14 perturbed
  cases GridKit already solved.
- **Agreement**: max |dV| < 0.01 pu and max |dTheta| < 1.0 deg between PM.jl and
  GridKit, across all cases - including the 4 stress cases flagged in
  [`pf_helper.md`](../work-notes/pf_helper.md) Section 13 (`load80pct`, `wind80pct`,
  `gen147off`, `gen10rand_off`) where GridKit's non-enforcement of generator
  reactive-power (Q) limits was hypothesized to matter most.

The verdict below is computed directly from `sweep_df` (Section 6) and the base-case
comparison (Section 5) - not asserted.


In [ ]:
DV_THRESH_PU = 0.01
DTHETA_THRESH_DEG = 1.0
STRESS_CASES = ["load80pct", "wind80pct", "gen147off", "gen10rand_off"]

n_total = len(sweep_df)
conv_df = sweep_df[sweep_df.pm_converged].copy()
n_converged = len(conv_df)
print(f"PM.jl converged on {n_converged}/{n_total} perturbed cases.")
if n_converged < n_total:
    print(
        f"  Non-converged cases: {sweep_df.loc[~sweep_df.pm_converged, 'case'].tolist()}"
    )

if n_converged:
    conv_df["agrees"] = (conv_df.max_dV_pu < DV_THRESH_PU) & (
        conv_df.max_dTheta_deg < DTHETA_THRESH_DEG
    )
    n_agree = conv_df.agrees.sum()
    print(
        f"\nCases meeting agreement thresholds (max|dV|<{DV_THRESH_PU} pu, "
        f"max|dTheta|<{DTHETA_THRESH_DEG} deg): {n_agree}/{n_converged}"
    )
    print(
        f"Overall max|dV|  across all converged cases: {conv_df.max_dV_pu.max():.6f} pu"
    )
    print(
        f"Overall max|dTheta| across all converged cases: {conv_df.max_dTheta_deg.max():.6f} deg"
    )

    stress_df = conv_df[conv_df.case.isin(STRESS_CASES)]
    print("\nStress cases (Section 13 discriminating experiments):")
    print(
        stress_df[["case", "max_dV_pu", "max_dTheta_deg", "agrees"]].to_string(
            index=False
        )
    )

    base_case_ok = (pm_vs_gk.dV.max() < DV_THRESH_PU) and (
        pm_vs_gk.dTheta.max() < DTHETA_THRESH_DEG
    )
    all_ok = base_case_ok and (n_agree == n_converged) and (n_converged == n_total)

    print()
    if all_ok:
        print(
            "VERDICT: GridKit's solve_pf agrees with PowerModels.jl within thresholds on "
            "the base case and all 14 perturbed cases. It is a reasonable PF solver for "
            "the Aleatoric UQ track's screening/development use, subject to re-checking "
            "this conclusion once real 8760-scenario cases are available (these 14 cases "
            "are synthetic stress tests, not the production PCM scenarios)."
        )
    else:
        print(
            "VERDICT: GridKit's solve_pf and PowerModels.jl diverge beyond the thresholds "
            "on at least one case above (see sweep_df / stress_df). Investigate whether "
            "this is due to Q-limit enforcement differences (pf_helper.md Section 13) "
            "before relying on solve_pf for the 8760-scenario production run; PM.jl may "
            "be required as the primary solver instead."
        )
else:
    print("\nVERDICT: inconclusive - PM.jl failed to converge on all perturbed cases.")